In [ ]:

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

from skmultilearn.problem_transform import BinaryRelevance, ClassifierChain

from sklearn.metrics import accuracy_score, hamming_loss, classification_report, f1_score

# Baixar recursos do NLTK
nltk.download('stopwords', quiet=True)
print("Ambiente e bibliotecas prontos.")

Ambiente e bibliotecas prontos.


In [11]:
# Célula 3: Carregamento e Limpeza dos Dados
# Carregar o dataset
df = pd.read_csv('train.csv')

# Definir colunas de rótulos
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Função de limpeza
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Aplicar a limpeza
print("Limpando texto...")
df['clean_comment_text'] = df['comment_text'].apply(clean_text)
print("Limpeza concluída.")
display(df[['comment_text', 'clean_comment_text']].head())

Limpando texto...
Limpeza concluída.


,comment_text,clean_comment_text
0,Explanation\nWhy the edits made under my usern...,explanation edits made username hardcore metal...
1,D'aww! He matches this background colour I'm s...,daww matches background colour im seemingly st...
2,"Hey man, I'm really not trying to edit war. It...",hey man im really trying edit war guy constant...
3,"""\nMore\nI can't make any real suggestions on ...",cant make real suggestions improvement wondere...
4,"You, sir, are my hero. Any chance you remember...",sir hero chance remember page thats


In [ ]:
# Vetorizador TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')

# Features (X) e Rótulos (y)
X = vectorizer.fit_transform(df['clean_comment_text'])
y = df[label_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dados prontos para modelagem:")
print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de X_test: {X_test.shape}")

Dados prontos para modelagem:
Formato de X_train: (127656, 10000)
Formato de X_test: (31915, 10000)


In [ ]:
results = {}
print("Estrutura de resultados criada. Pronta para armazenar o desempenho dos modelos.")

Estrutura de resultados criada. Pronta para armazenar o desempenho dos modelos.


In [ ]:
# Treinamento do Modelo 1 (BR + Logistic Regression)
print("--- Treinando Modelo 1: Binary Relevance com Regressão Logística ---")

# Define o classificador base
logreg = LogisticRegression(solver='liblinear', random_state=42)

# Define a estratégia multirrótulo
br_classifier = BinaryRelevance(classifier=logreg, require_dense=[False, True])

br_classifier.fit(X_train, y_train)

print("Treinamento do Modelo 1 concluído.")

--- Treinando Modelo 1: Binary Relevance com Regressão Logística ---
Treinamento do Modelo 1 concluído.


In [ ]:
#Avaliação do Modelo 1
print("--- Avaliando Modelo 1 ---")
predictions_br = br_classifier.predict(X_test)

acc = accuracy_score(y_test, predictions_br)
ham = hamming_loss(y_test, predictions_br)
f1_micro = f1_score(y_test, predictions_br, average='micro')
f1_macro = f1_score(y_test, predictions_br, average='macro')

results['BR + Logistic Regression'] = {'Subset Accuracy': acc, 'Hamming Loss': ham, 'F1 Micro': f1_micro, 'F1 Macro': f1_macro}

print(f"Subset Accuracy: {acc:.4f}")
print(f"Hamming Loss: {ham:.4f}")
print(f"F1-score (Micro): {f1_micro:.4f}")
print(f"F1-score (Macro): {f1_macro:.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, predictions_br, target_names=label_cols, zero_division=0))

--- Avaliando Modelo 1 ---
Subset Accuracy: 0.9186
Hamming Loss: 0.0196
F1-score (Micro): 0.6733
F1-score (Macro): 0.4896

Relatório de Classificação:
                precision    recall  f1-score   support

        toxic       0.90      0.61      0.73      3056
 severe_toxic       0.59      0.25      0.35       321
      obscene       0.91      0.63      0.74      1715
       threat       0.79      0.15      0.25        74
       insult       0.83      0.50      0.62      1614
identity_hate       0.73      0.15      0.24       294

    micro avg       0.88      0.55      0.67      7074
    macro avg       0.79      0.38      0.49      7074
 weighted avg       0.87      0.55      0.66      7074
  samples avg       0.05      0.05      0.05      7074



In [ ]:
# Treinamento do Modelo 2 (BR + LightGBM)
print("--- Treinando Modelo 2: Binary Relevance com LightGBM ---")

# Define o classificador base
# n_jobs=-1 usa todos os cores da CPU para acelerar
lgbm = LGBMClassifier(n_jobs=-1, random_state=42)

# Define a estratégia multirrótulo
br_lgbm_classifier = BinaryRelevance(classifier=lgbm, require_dense=[False, True])

br_lgbm_classifier.fit(X_train, y_train)

print("Treinamento do Modelo 2 concluído.")

--- Treinando Modelo 2: Binary Relevance com LightGBM ---
[LightGBM] [Info] Number of positive: 12238, number of negative: 115418
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 5.994963 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 572124
[LightGBM] [Info] Number of data points in the train set: 127656, number of used features: 9688
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.095867 -> initscore=-2.244014
[LightGBM] [Info] Start training from score -2.244014
[LightGBM] [Info] Number of positive: 1274, number of negative: 126382
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 4.827488 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 572124
[LightGBM] [Info] Number of data points in the train set: 127656, number of used features: 9688
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009980 -> initscore=-4.5971

In [ ]:
#  Avaliação do Modelo 2
print("--- Avaliando Modelo 2 ---")
predictions_lgbm = br_lgbm_classifier.predict(X_test)

acc = accuracy_score(y_test, predictions_lgbm)
ham = hamming_loss(y_test, predictions_lgbm)
f1_micro = f1_score(y_test, predictions_lgbm, average='micro')
f1_macro = f1_score(y_test, predictions_lgbm, average='macro')

results['BR + LightGBM'] = {'Subset Accuracy': acc, 'Hamming Loss': ham, 'F1 Micro': f1_micro, 'F1 Macro': f1_macro}

print(f"Subset Accuracy: {acc:.4f}")
print(f"Hamming Loss: {ham:.4f}")
print(f"F1-score (Micro): {f1_micro:.4f}")
print(f"F1-score (Macro): {f1_macro:.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, predictions_lgbm, target_names=label_cols, zero_division=0))

--- Avaliando Modelo 2 ---


/home/gabriel/gabriel/IA-PUC/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gabriel/gabriel/IA-PUC/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gabriel/gabriel/IA-PUC/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gabriel/gabriel/IA-PUC/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gabriel/gabriel/IA-PUC/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, b

Subset Accuracy: 0.9177
Hamming Loss: 0.0193
F1-score (Micro): 0.6948
F1-score (Macro): 0.5239

Relatório de Classificação:
                precision    recall  f1-score   support

        toxic       0.89      0.61      0.72      3056
 severe_toxic       0.45      0.18      0.26       321
      obscene       0.88      0.72      0.79      1715
       threat       0.47      0.27      0.34        74
       insult       0.76      0.60      0.67      1614
identity_hate       0.61      0.25      0.36       294

    micro avg       0.83      0.60      0.69      7074
    macro avg       0.68      0.44      0.52      7074
 weighted avg       0.82      0.60      0.69      7074
  samples avg       0.05      0.05      0.05      7074



In [ ]:
print("--- Treinando Modelo 3: Classifier Chain com Regressão Logística ---")

# Define o classificador base (o mesmo do baseline)
logreg_chain = LogisticRegression(solver='liblinear', random_state=42)

# Define a estratégia multirrótulo
# A ordem dos rótulos pode impactar o resultado
chain_classifier = ClassifierChain(classifier=logreg_chain, require_dense=[False, True])

chain_classifier.fit(X_train, y_train)

print("Treinamento do Modelo 3 concluído.")

--- Treinando Modelo 3: Classifier Chain com Regressão Logística ---
Treinamento do Modelo 3 concluído.


In [ ]:
print("--- Avaliando Modelo 3 ---")
predictions_chain = chain_classifier.predict(X_test)

acc = accuracy_score(y_test, predictions_chain)
ham = hamming_loss(y_test, predictions_chain)
f1_micro = f1_score(y_test, predictions_chain, average='micro')
f1_macro = f1_score(y_test, predictions_chain, average='macro')

results['Chain + Logistic Regression'] = {'Subset Accuracy': acc, 'Hamming Loss': ham, 'F1 Micro': f1_micro, 'F1 Macro': f1_macro}

print(f"Subset Accuracy: {acc:.4f}")
print(f"Hamming Loss: {ham:.4f}")
print(f"F1-score (Micro): {f1_micro:.4f}")
print(f"F1-score (Macro): {f1_macro:.4f}")
print("\nRelatório de Classificação:\n", classification_report(y_test, predictions_chain, target_names=label_cols, zero_division=0))

--- Avaliando Modelo 3 ---
Subset Accuracy: 0.9200
Hamming Loss: 0.0193
F1-score (Micro): 0.6916
F1-score (Macro): 0.5066

Relatório de Classificação:
                precision    recall  f1-score   support

        toxic       0.90      0.61      0.73      3056
 severe_toxic       0.59      0.19      0.29       321
      obscene       0.88      0.69      0.78      1715
       threat       0.71      0.16      0.26        74
       insult       0.74      0.60      0.66      1614
identity_hate       0.86      0.20      0.33       294

    micro avg       0.85      0.58      0.69      7074
    macro avg       0.78      0.41      0.51      7074
 weighted avg       0.84      0.58      0.68      7074
  samples avg       0.05      0.05      0.05      7074



In [ ]:
results_df = pd.DataFrame.from_dict(results, orient='index')

results_df_sorted = results_df.sort_values(by='Hamming Loss', ascending=True)

print("--- Tabela Comparativa de Desempenho dos Modelos ---")
display(results_df_sorted)

--- Tabela Comparativa de Desempenho dos Modelos ---


,Subset Accuracy,Hamming Loss,F1 Micro,F1 Macro
Chain + Logistic Regression,0.920038,0.019259,0.691587,0.506576
BR + LightGBM,0.917656,0.019348,0.694836,0.523892
BR + Logistic Regression,0.918565,0.019594,0.673341,0.489572
